# 🔍 Retrieval-Augmented Generation (RAG)
### Session 4 — From Naive Search to Production-Grade Q&A

> **Prerequisites:** Sessions 1–3. You should already have document chunks embedded and stored in a vector store.

---

## Why RAG?

LLMs have two fundamental limitations:

| Problem | Symptom | RAG Solution |
|---|---|---|
| **Knowledge cutoff** | "I don't have information after [date]" | Ground answers in live documents |
| **Hallucination** | Confident but wrong answers | Force model to cite real sources |
| **No private data** | Can't answer about internal docs | Inject your documents as context |

RAG bridges the gap between a frozen LLM and the real world by **retrieving relevant context** and injecting it into the prompt before generating an answer.

## What You'll Build

```
  Naive RAG  →  Query Rewriting  →  Hybrid Search  →  Re-Ranking  →  Grounded Generation  →  Evaluation
```

**Sections:**
1. 🏗️ Naive RAG — the foundation
2. ✍️ Query Rewriting & Expansion
3. 🧪 HyDE — Hypothetical Document Embeddings
4. 🔀 Hybrid Search — BM25 + Vector with RRF
5. 🏆 Re-Ranking — Cross-Encoder scoring
6. 📜 Grounded Generation — Citations & Source Attribution
7. 📊 RAG Evaluation — Faithfulness, Relevance, Precision
8. 🏭 Production RAG Pipeline — everything assembled

---

## 📦 Setup

In [ ]:
!pip install openai numpy rank-bm25 --quiet

import os, json, re, uuid, math
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Optional
import numpy as np
from openai import OpenAI
from rank_bm25 import BM25Okapi

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# ── Core helpers ──────────────────────────────────────────────────────────────
def chat(messages: list[dict], model="gpt-4o-mini", temperature=0) -> str:
    resp = client.chat.completions.create(
        model=model, temperature=temperature, messages=messages)
    return resp.choices[0].message.content

def ask(prompt: str, system="You are a helpful assistant.",
        model="gpt-4o-mini", temperature=0) -> str:
    return chat([{"role": "system", "content": system},
                 {"role": "user",   "content": prompt}],
                model=model, temperature=temperature)

def embed(text: str) -> list[float]:
    resp = client.embeddings.create(input=text, model="text-embedding-3-small")
    return resp.data[0].embedding

def cosine_sim(a, b) -> float:
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print("✅ Setup complete!")

In [ ]:
# ── Knowledge Base ─────────────────────────────────────────────────────────────
# A compact but realistic document collection that covers multiple topics.
# Each document simulates what you'd get after Session 3 document processing.

DOCUMENTS = [
    # ── AI Platform Technical Spec ────────────────────────────────────────────
    {"id": "tech_01", "source": "ai_platform_spec.pdf", "section": "Overview",
     "text": "The AI Platform v2.0 is a cloud-native system for deploying large language model applications. It requires Python 3.10 or higher, a minimum of 16 GB RAM (32 GB recommended), and optional CUDA-compatible GPU for faster inference. The platform targets production deployment in Q3 2025."},

    {"id": "tech_02", "source": "ai_platform_spec.pdf", "section": "API Endpoints",
     "text": "The platform exposes four REST API endpoints: POST /v1/embed for generating vector embeddings, POST /v1/search for performing semantic search, POST /v1/extract for LLM-based structured data extraction, and POST /v1/summarize for document summarization. All endpoints require a valid Bearer token."},

    {"id": "tech_03", "source": "ai_platform_spec.pdf", "section": "Data Retention",
     "text": "All vector embeddings stored in Amazon S3 Vectors are retained for 90 days by default. Administrators can configure custom retention windows between 7 and 365 days via the management console. Data is encrypted at rest using AES-256 and in transit using TLS 1.3."},

    {"id": "tech_04", "source": "ai_platform_spec.pdf", "section": "Pricing",
     "text": "The AI Platform is priced based on usage. Embedding generation costs $0.002 per 1,000 tokens. Semantic search queries are billed at $0.001 per query. Storage in S3 Vectors costs $0.023 per GB-month. There is a free tier of 10,000 embeddings and 5,000 search queries per month per account."},

    # ── Q1 Sales Report ───────────────────────────────────────────────────────
    {"id": "sales_01", "source": "q1_2025_report.docx", "section": "Executive Summary",
     "text": "Total revenue for Q1 2025 reached PHP 12.4 million, representing a 23% year-over-year growth compared to Q1 2024. Customer acquisition increased by 35% in the Southeast Asia region, driven by new partnerships in Indonesia and Vietnam."},

    {"id": "sales_02", "source": "q1_2025_report.docx", "section": "Regional Breakdown",
     "text": "Regional performance in Q1 2025: Philippines contributed PHP 5.2M with 28% YoY growth, Indonesia contributed PHP 4.1M with 19% YoY growth, and Vietnam contributed PHP 3.1M with 22% YoY growth. The Philippines remains the top revenue contributor."},

    {"id": "sales_03", "source": "q1_2025_report.docx", "section": "Key Highlights",
     "text": "Key operational highlights for Q1 2025: Three new product SKUs were launched in February. Customer retention rate improved to 87%, up from 81% in Q1 2024. Operating costs were reduced by 8% through process automation and vendor renegotiations."},

    # ── Meeting Notes ─────────────────────────────────────────────────────────
    {"id": "meet_01", "source": "product_roadmap_meeting.txt", "section": "Decisions",
     "text": "Product roadmap meeting held April 15, 2025, attended by CEO Rosa, CTO Marco, and Product Lead Lia. Key decisions: approved PHP 2.5 million budget for AI feature development, target launch date for v2.0 set to September 2025, Lia assigned to lead the new personalization engine project."},

    {"id": "meet_02", "source": "product_roadmap_meeting.txt", "section": "Action Items",
     "text": "Action items from April 15 meeting: Marco must prepare technical specification document by April 30. Rosa must align with the board on the revised product timeline. Lia must hire two additional ML engineers by May 15 to staff the personalization engine team."},

    {"id": "meet_03", "source": "product_roadmap_meeting.txt", "section": "Risks",
     "text": "Two risks were identified at the April 15 meeting. First, dependency on a third-party data provider is rated as medium risk. Second, potential delay if ML engineers are not onboarded by May 15 is rated as high risk, as it directly impacts the September v2.0 launch date."},

    # ── HR / Employee Data ────────────────────────────────────────────────────
    {"id": "hr_01", "source": "employees.xlsx", "section": "Engineering Team",
     "text": "Engineering department employees: Ana Reyes is a Senior Engineer earning PHP 85,000/month, joined March 2021. Maria Cruz is a Data Scientist earning PHP 90,000/month, joined November 2020. Both report to CTO Marco."},

    {"id": "hr_02", "source": "employees.xlsx", "section": "Other Departments",
     "text": "Other department employees: Juan Santos is Brand Manager in Marketing earning PHP 72,000/month (joined July 2022). Paolo Delos is Account Executive in Sales earning PHP 65,000/month (joined January 2023). Lea Gomez is HR Manager earning PHP 70,000/month (joined June 2019)."},

    # ── Company Policy ────────────────────────────────────────────────────────
    {"id": "policy_01", "source": "hr_policies.pdf", "section": "Leave Policy",
     "text": "All regular employees are entitled to 15 vacation leave credits per year, 15 sick leave credits per year, and 5 emergency leave credits. Leave credits are non-cumulative and expire at the end of each calendar year. Unused vacation leave may be converted to cash at 50% of daily rate."},

    {"id": "policy_02", "source": "hr_policies.pdf", "section": "Remote Work Policy",
     "text": "The company operates on a hybrid work model. Employees are required to be in the office a minimum of 3 days per week (Monday, Wednesday, Friday by default). Full remote work arrangements require VP-level approval and are reviewed quarterly. Remote workers must maintain a reliable internet connection of at least 25 Mbps."},
]

print(f"✅ Knowledge base loaded: {len(DOCUMENTS)} documents across "
      f"{len(set(d['source'] for d in DOCUMENTS))} sources")

In [ ]:
# ── Pre-compute embeddings for all documents (do this once) ───────────────────
print("Embedding knowledge base...")
for doc in DOCUMENTS:
    doc["embedding"] = embed(doc["text"])
    print(f"  [{doc['id']}] ✓")

print(f"\n✅ {len(DOCUMENTS)} embeddings ready. Embedding dim: {len(DOCUMENTS[0]['embedding'])}")

---

## 🏗️ Section 1: Naive RAG — The Foundation

The simplest RAG pipeline has three steps:

```
User Question
    ↓  embed
Query Vector
    ↓  cosine similarity vs. all document vectors
Top-K Chunks
    ↓  inject into prompt
LLM Answer
```

This is the baseline we will improve upon throughout the notebook.

In [ ]:
def vector_search(query: str, docs: list[dict], top_k: int = 4) -> list[dict]:
    """Pure semantic search using cosine similarity."""
    q_emb = embed(query)
    scored = [(cosine_sim(q_emb, d["embedding"]), d) for d in docs]
    scored.sort(key=lambda x: x[0], reverse=True)
    return [{**d, "score": s, "retrieval_method": "vector"}
            for s, d in scored[:top_k]]


def build_context(results: list[dict]) -> str:
    """Format retrieved chunks into a context block for the LLM."""
    parts = []
    for i, r in enumerate(results, 1):
        parts.append(
            f"[{i}] Source: {r['source']} | Section: {r['section']}\n"
            f"{r['text']}"
        )
    return "\n\n".join(parts)


def naive_rag(question: str, docs: list[dict] = DOCUMENTS,
              top_k: int = 3) -> dict:
    """The simplest possible RAG pipeline."""
    # 1. Retrieve
    results = vector_search(question, docs, top_k=top_k)

    # 2. Build context
    context = build_context(results)

    # 3. Generate
    prompt = f"""Answer the question using ONLY the provided context.
If the answer is not in the context, say "I don't have that information."

Context:
{context}

Question: {question}
Answer:"""

    answer = ask(prompt)
    return {"question": question, "answer": answer,
            "retrieved": results, "context": context}


# ── Test naive RAG ─────────────────────────────────────────────────────────────
test_questions = [
    "What is the total Q1 2025 revenue and how does it compare to last year?",
    "Who is responsible for hiring ML engineers and by when?",
    "How much does embedding generation cost?",
]

for q in test_questions:
    result = naive_rag(q)
    print(f"Q: {q}")
    print(f"A: {result['answer']}")
    print(f"   Sources: {[r['id'] for r in result['retrieved']]}")
    print()

In [ ]:
# ── Where naive RAG breaks down ───────────────────────────────────────────────
# These queries expose weaknesses that the later sections will fix.

failure_cases = [
    # Keyword mismatch: user says "ML engineers salary" but doc says "PHP 90,000"
    "What is the pay for the data science team?",
    # Vague pronoun / missing context
    "What did they decide about the budget?",
    # Exact product code — embeddings may miss this
    "What is the cost of POST /v1/search?",
]

print("=" * 60)
print("NAIVE RAG FAILURE CASES")
print("=" * 60)
for q in failure_cases:
    result = naive_rag(q)
    top_source = result['retrieved'][0]['id'] if result['retrieved'] else 'none'
    print(f"Q: {q}")
    print(f"A: {result['answer'][:200]}")
    print(f"   Top retrieved: {top_source}")
    print()

---

## ✍️ Section 2: Query Rewriting & Expansion

Users rarely phrase questions the way documents are written. Query rewriting transforms the question into forms more likely to match your document vocabulary.

### Three Query Rewriting Strategies

| Strategy | What it does | Best for |
|---|---|---|
| **Rewrite** | Rephrase the query more clearly | Vague or ambiguous questions |
| **Multi-query** | Generate N alternate phrasings | Broad queries needing wide coverage |
| **Step-back** | Ask a more general question first | Specific queries that need context |

After rewriting, results from all variants are merged and de-duplicated.

In [ ]:
def rewrite_query(query: str) -> str:
    """Rephrase a potentially vague query into a precise search query."""
    prompt = f"""Rewrite the following user question as a clear, concise search query.
Remove vague pronouns, add specificity, and use vocabulary likely to appear in documents.
Return ONLY the rewritten query — no explanation.

Original question: {query}
Rewritten query:"""
    return ask(prompt).strip()


def generate_multi_query(query: str, n: int = 3) -> list[str]:
    """Generate N alternate phrasings of a query to cast a wider retrieval net."""
    prompt = f"""Generate {n} different search queries that would retrieve information
to answer the following question. Each query should approach the topic differently.
Return ONLY the queries, one per line, no numbering or extra text.

Question: {query}"""
    raw = ask(prompt)
    queries = [q.strip() for q in raw.strip().split("\n") if q.strip()]
    return queries[:n]


def step_back_query(query: str) -> str:
    """Generate a broader, more general version of the query for better context."""
    prompt = f"""Given this specific question, generate a more general background question
that provides context needed to answer the specific one.
Return ONLY the general question.

Specific question: {query}
General background question:"""
    return ask(prompt).strip()


# ── Demo all three strategies ─────────────────────────────────────────────────
test_q = "What did they decide about the budget?"

print(f"Original:     {test_q}")
print(f"Rewritten:    {rewrite_query(test_q)}")
print(f"Step-back:    {step_back_query(test_q)}")
print("Multi-query:")
for q in generate_multi_query(test_q, n=3):
    print(f"  • {q}")

In [ ]:
def dedup_results(all_results: list[list[dict]]) -> list[dict]:
    """Merge results from multiple queries, keeping the best score per document."""
    best = {}  # doc_id → best result
    for result_set in all_results:
        for r in result_set:
            doc_id = r["id"]
            if doc_id not in best or r["score"] > best[doc_id]["score"]:
                best[doc_id] = r
    return sorted(best.values(), key=lambda x: x["score"], reverse=True)


def query_expansion_rag(question: str, docs: list[dict] = DOCUMENTS,
                        top_k: int = 3) -> dict:
    """RAG with multi-query expansion for broader retrieval coverage."""
    # Generate original + alternates
    rewritten = rewrite_query(question)
    alternates = generate_multi_query(question, n=2)
    all_queries = [rewritten] + alternates

    print(f"  Searching with {len(all_queries)} query variants")

    # Retrieve for each query
    all_results = [vector_search(q, docs, top_k=top_k) for q in all_queries]

    # Merge and de-duplicate
    merged = dedup_results(all_results)[:top_k]

    # Generate
    context = build_context(merged)
    prompt = f"""Answer the question using ONLY the provided context.
If the answer is not in the context, say "I don't have that information."

Context:
{context}

Question: {question}
Answer:"""

    return {"question": question, "queries_used": all_queries,
            "answer": ask(prompt), "retrieved": merged}


# ── Compare naive vs. query expansion ────────────────────────────────────────
q = "What did they decide about the budget?"

print("NAIVE RAG:")
naive = naive_rag(q)
print(f"  Answer: {naive['answer']}")
print(f"  Sources: {[r['id'] for r in naive['retrieved']]}")

print("\nQUERY EXPANSION RAG:")
expanded = query_expansion_rag(q)
print(f"  Answer: {expanded['answer']}")
print(f"  Sources: {[r['id'] for r in expanded['retrieved']]}")

---

## 🧪 Section 3: HyDE — Hypothetical Document Embeddings

**The core insight:** a short question and a long document paragraph exist in different parts of the embedding space. The question sounds like a question; the document sounds like an answer.

**HyDE** (Gao et al., 2022) solves this by generating a *hypothetical* answer first, then using that answer's embedding for retrieval — doing document-to-document comparison instead of question-to-document.

```
Standard RAG:  question → embed(question) → search → documents
                          ❌ question and docs live in different regions

HyDE:          question → LLM → hypothetical_answer
                              → embed(hypothetical_answer) → search → documents  
                                ✅ answer and docs are in the same region
```

> The hypothetical answer doesn't need to be factually correct — it just needs to look like a document.

In [ ]:
def generate_hypothetical_document(question: str) -> str:
    """
    Generate a hypothetical document passage that would answer the question.
    This document may not be factually accurate — it's used only for retrieval.
    """
    prompt = f"""Write a short document passage (2-4 sentences) that would directly answer
the following question. Write it as a factual excerpt from an internal business document.
Do NOT start with phrases like 'The answer is' or 'According to'. Just write the passage.

Question: {question}
Document passage:"""
    return ask(prompt, temperature=0.3)


def hyde_search(question: str, docs: list[dict], top_k: int = 4) -> dict:
    """Retrieve using a hypothetical document embedding instead of the raw query."""
    hypo_doc = generate_hypothetical_document(question)
    results = vector_search(hypo_doc, docs, top_k=top_k)
    for r in results:
        r["retrieval_method"] = "hyde"
    return {"hypothetical_document": hypo_doc, "results": results}


def hyde_rag(question: str, docs: list[dict] = DOCUMENTS,
             top_k: int = 3) -> dict:
    """Full RAG pipeline using HyDE retrieval."""
    hyde_output = hyde_search(question, docs, top_k=top_k)
    results = hyde_output["results"]
    context = build_context(results)

    prompt = f"""Answer the question using ONLY the provided context.
If the answer is not in the context, say "I don't have that information."

Context:
{context}

Question: {question}
Answer:"""

    return {"question": question,
            "hypothetical_document": hyde_output["hypothetical_document"],
            "answer": ask(prompt), "retrieved": results}


# ── HyDE demo ─────────────────────────────────────────────────────────────────
q = "What is the pay for the data science team?"

hypo = generate_hypothetical_document(q)
print("Question:")
print(f"  {q}")
print("\nHypothetical document generated by LLM:")
print(f"  {hypo}")

In [ ]:
# ── Compare naive vs HyDE on keyword-mismatch queries ─────────────────────────
hard_queries = [
    "What is the pay for the data science team?",   # 'pay' vs 'salary' / 'earning'
    "When can the ML hiring goal be completed?",     # 'goal' vs 'action item' / 'deadline'
]

for q in hard_queries:
    print("=" * 65)
    print(f"Question: {q}")

    naive  = naive_rag(q, top_k=2)
    hyde   = hyde_rag(q, top_k=2)

    print(f"\n  NAIVE  → [{', '.join(r['id'] for r in naive['retrieved'])}]")
    print(f"    {naive['answer'][:150]}")
    print(f"\n  HyDE   → [{', '.join(r['id'] for r in hyde['retrieved'])}]")
    print(f"    {hyde['answer'][:150]}")
    print()

---

## 🔀 Section 4: Hybrid Search — BM25 + Vector with RRF

### The Problem with Vector-Only Search

Semantic embeddings are great at capturing meaning, but they can miss **exact matches** — product codes, error codes, names, URLs, or specific numbers. BM25 (keyword search) excels at these.

| Query Type | Vector | BM25 |
|---|---|---|
| "What is machine learning?" | ✅ Great | ⚠️ Misses synonyms |
| "What does POST /v1/search cost?" | ⚠️ May miss exact path | ✅ Exact token match |
| "PHP 90,000 employee" | ⚠️ May generalize salary | ✅ Exact number match |

### Reciprocal Rank Fusion (RRF)

We can't just average BM25 and cosine scores — they live on different scales. RRF sidesteps this by using **ranks** instead of raw scores:

$$\text{RRF}(d) = \sum_{r \in R} \frac{1}{k + \text{rank}_r(d)}$$

Where $k$ is a smoothing constant (typically 60). A document ranked #1 in both systems gets a very high RRF score regardless of its raw BM25 or cosine values.

In [ ]:
# ── Build BM25 index ──────────────────────────────────────────────────────────
def tokenize(text: str) -> list[str]:
    """Simple tokenizer: lowercase, split on non-alphanumeric."""
    return re.findall(r'[a-z0-9./]+', text.lower())

# Build the BM25 corpus
corpus_texts = [d["text"] for d in DOCUMENTS]
tokenized_corpus = [tokenize(t) for t in corpus_texts]
bm25_index = BM25Okapi(tokenized_corpus)

print(f"✅ BM25 index built over {len(tokenized_corpus)} documents")
print(f"   Avg tokens per doc: {np.mean([len(t) for t in tokenized_corpus]):.1f}")


def bm25_search(query: str, docs: list[dict], top_k: int = 4) -> list[dict]:
    """Keyword-based BM25 retrieval."""
    tokens = tokenize(query)
    scores = bm25_index.get_scores(tokens)
    ranked = np.argsort(scores)[::-1][:top_k]
    return [{**docs[i], "score": float(scores[i]), "retrieval_method": "bm25"}
            for i in ranked if scores[i] > 0]


# Test BM25 on an exact-match query
q = "What does POST /v1/search cost?"
print(f"\nBM25 results for: '{q}'")
for r in bm25_search(q, DOCUMENTS, top_k=3):
    print(f"  [{r['score']:.3f}] {r['id']}: {r['text'][:80]}...")

In [ ]:
def reciprocal_rank_fusion(result_lists: list[list[dict]],
                           k: int = 60) -> list[dict]:
    """
    Merge multiple ranked result lists using Reciprocal Rank Fusion.
    
    RRF(d) = Σ 1 / (k + rank(d))  for each retriever that returned d
    
    k=60 is the standard default — smaller k favors top-ranked items more strongly.
    """
    rrf_scores: dict[str, float] = defaultdict(float)
    doc_map: dict[str, dict] = {}

    for result_list in result_lists:
        for rank, doc in enumerate(result_list, start=1):
            doc_id = doc["id"]
            rrf_scores[doc_id] += 1.0 / (k + rank)
            if doc_id not in doc_map:
                doc_map[doc_id] = doc

    # Return sorted by RRF score
    sorted_ids = sorted(rrf_scores, key=lambda x: rrf_scores[x], reverse=True)
    return [{**doc_map[doc_id], "rrf_score": rrf_scores[doc_id],
             "score": rrf_scores[doc_id], "retrieval_method": "hybrid_rrf"}
            for doc_id in sorted_ids]


def hybrid_search(query: str, docs: list[dict], top_k: int = 4,
                  rrf_k: int = 60) -> list[dict]:
    """Combine semantic + keyword search using RRF."""
    vector_results = vector_search(query, docs, top_k=top_k * 2)
    bm25_results   = bm25_search(query, docs, top_k=top_k * 2)
    fused = reciprocal_rank_fusion([vector_results, bm25_results], k=rrf_k)
    return fused[:top_k]


def hybrid_rag(question: str, docs: list[dict] = DOCUMENTS,
               top_k: int = 3) -> dict:
    """RAG with hybrid BM25 + vector search and RRF fusion."""
    results = hybrid_search(question, docs, top_k=top_k)
    context = build_context(results)
    prompt = f"""Answer the question using ONLY the provided context.
If the answer is not in the context, say "I don't have that information."

Context:
{context}

Question: {question}
Answer:"""
    return {"question": question, "answer": ask(prompt), "retrieved": results}


# ── Side-by-side comparison ───────────────────────────────────────────────────
comparison_queries = [
    "What does POST /v1/search cost?",
    "What is the leave policy?",
    "Who is Maria Cruz and what is her salary?",
]

for q in comparison_queries:
    print(f"{'='*65}")
    print(f"Query: {q}")

    v_results = vector_search(q, DOCUMENTS, top_k=3)
    b_results = bm25_search(q, DOCUMENTS, top_k=3)
    h_results = hybrid_search(q, DOCUMENTS, top_k=3)

    print(f"  Vector: {[r['id'] for r in v_results]}")
    print(f"  BM25:   {[r['id'] for r in b_results]}")
    print(f"  Hybrid: {[r['id'] for r in h_results]} ← RRF fusion")
    print()

In [ ]:
# ── Visualize how RRF works ───────────────────────────────────────────────────
def explain_rrf(query: str, docs: list[dict], k: int = 60):
    """Show how RRF combines two ranked lists step by step."""
    v_results = vector_search(query, docs, top_k=5)
    b_results = bm25_search(query, docs, top_k=5)
    fused = reciprocal_rank_fusion([v_results, b_results], k=k)

    v_rank = {r['id']: i+1 for i, r in enumerate(v_results)}
    b_rank = {r['id']: i+1 for i, r in enumerate(b_results)}

    print(f"Query: '{query}'  (k={k})")
    print(f"{'Doc':<12} {'Vec rank':>9} {'BM25 rank':>10} "
          f"{'Vec RRF':>9} {'BM25 RRF':>9} {'Total':>8}")
    print("─" * 60)
    for r in fused[:5]:
        did = r['id']
        vr = v_rank.get(did, '—')
        br = b_rank.get(did, '—')
        v_contrib = 1/(k + vr) if isinstance(vr, int) else 0
        b_contrib = 1/(k + br) if isinstance(br, int) else 0
        vrs = str(vr)
        brs = str(br)
        print(f"  {did:<10} {vrs:>9} {brs:>10} "
              f"{v_contrib:>9.4f} {b_contrib:>9.4f} {r['rrf_score']:>8.4f}")

explain_rrf("What does POST /v1/search cost?", DOCUMENTS)

---

## 🏆 Section 5: Re-Ranking

Retrieval (BM25, vector, hybrid) is fast but approximate. **Re-ranking** applies a more expensive but more accurate model to the top-K retrieved candidates to re-order them before sending to the LLM.

```
Retrieval  →  top-20 candidates  →  Re-ranker  →  top-3  →  LLM
  (fast)                           (expensive,        (precise)
                                    accurate)
```

### Two Re-Ranking Approaches

| Method | How | Cost | Accuracy |
|---|---|---|---|
| **Cross-encoder (neural)** | Score each (query, doc) pair jointly | Medium | Very high |
| **LLM-as-judge** | LLM rates relevance 0–10 | High (slow) | High |

We implement LLM-as-judge here (no additional model needed).

In [ ]:
def llm_rerank(question: str, candidates: list[dict], top_k: int = 3) -> list[dict]:
    """
    Re-rank retrieved candidates using an LLM as the judge.
    Each candidate is scored 0–10 for relevance to the question.
    This is more accurate than embedding similarity for complex queries.
    """
    if not candidates:
        return []

    # Build the scoring prompt
    passages_text = ""
    for i, c in enumerate(candidates):
        passages_text += f"[{i}] {c['text']}\n\n"

    prompt = f"""You are a relevance judge. Score each passage's relevance to the question.
Return ONLY a JSON array of scores, one per passage, integers from 0 (irrelevant) to 10 (perfectly relevant).
Example output: [8, 3, 6, 1, 9]

Question: {question}

Passages:
{passages_text}
Scores (JSON array only):"""

    raw = ask(prompt)
    try:
        scores = json.loads(re.search(r'\[.*?\]', raw, re.DOTALL).group())
    except Exception:
        # Fallback: use original order
        return candidates[:top_k]

    # Attach scores and sort
    scored = []
    for i, (c, s) in enumerate(zip(candidates, scores)):
        scored.append({**c, "rerank_score": s, "original_rank": i + 1})

    scored.sort(key=lambda x: x["rerank_score"], reverse=True)
    return scored[:top_k]


def reranked_rag(question: str, docs: list[dict] = DOCUMENTS,
                 retrieve_k: int = 8, final_k: int = 3) -> dict:
    """
    Full pipeline: Hybrid search → LLM re-ranking → Generation.
    Retrieve more (retrieve_k), then re-rank down to final_k.
    """
    # Stage 1: Broad retrieval
    candidates = hybrid_search(question, docs, top_k=retrieve_k)
    print(f"  Retrieved {len(candidates)} candidates")

    # Stage 2: Re-rank
    reranked = llm_rerank(question, candidates, top_k=final_k)
    print(f"  Re-ranked to top {len(reranked)}")

    # Stage 3: Generate
    context = build_context(reranked)
    prompt = f"""Answer the question using ONLY the provided context.
If the answer is not in the context, say "I don't have that information."

Context:
{context}

Question: {question}
Answer:"""

    return {"question": question, "answer": ask(prompt),
            "retrieved": reranked, "candidates_count": len(candidates)}


# ── Demo re-ranking ───────────────────────────────────────────────────────────
q = "What are the risks around the September product launch?"
print(f"Question: {q}\n")

# Show before/after re-ranking
candidates = hybrid_search(q, DOCUMENTS, top_k=6)
print("Before re-ranking (hybrid order):")
for r in candidates:
    print(f"  [{r['rrf_score']:.4f} RRF] {r['id']}: {r['text'][:70]}...")

reranked = llm_rerank(q, candidates, top_k=3)
print("\nAfter LLM re-ranking:")
for r in reranked:
    print(f"  [score {r['rerank_score']}/10] (was rank #{r['original_rank']}) {r['id']}: {r['text'][:70]}...")

---

## 📜 Section 6: Grounded Generation — Citations & Source Attribution

A RAG system is only trustworthy if users can verify its answers. **Grounded generation** forces the LLM to:
1. Answer strictly from context (no hallucination)
2. Cite the specific source for each claim
3. Acknowledge gaps instead of making things up

This is what separates a toy RAG demo from a production-grade system.

In [ ]:
@dataclass
class GroundedAnswer:
    answer: str
    citations: list[dict]
    has_gaps: bool
    confidence: str  # "high", "medium", "low"
    raw_json: dict = field(default_factory=dict)


def grounded_generate(question: str, retrieved_docs: list[dict]) -> GroundedAnswer:
    """
    Generate an answer with explicit source citations.
    Forces the LLM to:
      - Cite the document ID for each factual claim
      - Flag if information is incomplete
      - Report confidence level
    """
    # Build numbered context
    context_parts = []
    for i, doc in enumerate(retrieved_docs):
        context_parts.append(
            f"[DOC-{i+1}] Source: {doc['source']} | Section: {doc['section']}\n"
            f"Content: {doc['text']}"
        )
    context = "\n\n".join(context_parts)

    prompt = f"""
You are a precise document analyst. Answer ONLY from the provided documents.

RULES:
1. Use [DOC-N] inline citations for every factual claim.
2. If you cannot fully answer from the documents, list what is missing.
3. Never add information not in the documents.

Return a JSON object with this exact structure:
{{
  "answer": "Full answer with inline [DOC-N] citations",
  "citations": [
    {{"ref": "DOC-1", "source": "filename", "section": "section name", "claim": "the specific claim supported"}}
  ],
  "has_gaps": true or false,
  "gaps": "Description of what information is missing, or null",
  "confidence": "high|medium|low"
}}

Documents:
{context}

Question: {question}
"""

    raw = ask(prompt)
    try:
        clean = re.sub(r"```json|```", "", raw).strip()
        data = json.loads(clean)
    except Exception:
        return GroundedAnswer(answer=raw, citations=[], has_gaps=True,
                              confidence="low")

    return GroundedAnswer(
        answer=data.get("answer", ""),
        citations=data.get("citations", []),
        has_gaps=data.get("has_gaps", False),
        confidence=data.get("confidence", "low"),
        raw_json=data
    )


def grounded_rag(question: str, docs: list[dict] = DOCUMENTS,
                 top_k: int = 4) -> GroundedAnswer:
    """Full grounded RAG: hybrid retrieval + cited generation."""
    results = hybrid_search(question, docs, top_k=top_k)
    return grounded_generate(question, results)


# ── Demo grounded generation ──────────────────────────────────────────────────
q = "What are the data retention and encryption standards for the AI platform?"
result = grounded_rag(q)

print(f"Q: {q}\n")
print(f"Answer: {result.answer}\n")
print(f"Confidence: {result.confidence}")
print(f"Has gaps: {result.has_gaps}")
print(f"\nCitations ({len(result.citations)}):")
for c in result.citations:
    print(f"  {c['ref']} → {c['source']} [{c['section']}]")
    print(f"    Claim: {c['claim'][:100]}")

In [ ]:
# ── Test with a question the docs can't fully answer ─────────────────────────
q_partial = "What is the company's data privacy policy and GDPR compliance status?"
result = grounded_rag(q_partial)

print(f"Q: {q_partial}\n")
print(f"Answer: {result.answer}")
print(f"\nConfidence: {result.confidence}")
print(f"Has gaps: {result.has_gaps}")
if result.has_gaps:
    print(f"Missing info: {result.raw_json.get('gaps')}")

---

## 📊 Section 7: RAG Evaluation

You can't improve what you can't measure. RAG evaluation uses **LLM-as-judge** metrics to automatically score pipeline quality without requiring ground-truth human annotations for every query.

### The Four Core Metrics

| Metric | Measures | Formula |
|---|---|---|
| **Faithfulness** | Is the answer grounded in the retrieved context? | claims_supported / total_claims |
| **Answer Relevance** | Does the answer address the question? | 0–1 LLM score |
| **Context Precision** | Are retrieved docs relevant to the question? | relevant_retrieved / total_retrieved |
| **Context Recall** | Did we retrieve all necessary information? | 0–1 LLM score |

> These metrics are inspired by the [RAGAS framework](https://docs.ragas.io) but implemented from scratch here so you understand what's happening under the hood.

In [ ]:
@dataclass
class RAGEvalResult:
    question: str
    answer: str
    faithfulness: float        # 0-1: answer supported by context
    answer_relevance: float    # 0-1: answer addresses the question
    context_precision: float   # 0-1: fraction of retrieved docs that are relevant
    context_recall: float      # 0-1: context covers what's needed
    overall: float             # harmonic mean of the four

    def __str__(self):
        return (f"Q: {self.question[:60]}...\n"
                f"  Faithfulness:    {self.faithfulness:.2f}\n"
                f"  Ans. Relevance:  {self.answer_relevance:.2f}\n"
                f"  Context Prec.:   {self.context_precision:.2f}\n"
                f"  Context Recall:  {self.context_recall:.2f}\n"
                f"  ─────────────────────────\n"
                f"  Overall Score:   {self.overall:.2f}")


def eval_faithfulness(answer: str, context: str) -> float:
    """What fraction of answer claims are supported by the context? (0–1)"""
    prompt = f"""
Evaluate faithfulness: how well is the answer supported by the context?

Context: {context[:2000]}
Answer: {answer}

Instructions:
1. List each factual claim in the answer (max 6).
2. For each claim, determine: Supported / Not Supported / Not Verifiable.
3. Return ONLY JSON: {{"claims": [{{"claim": "text", "verdict": "Supported"}}, ...], "score": 0.85}}
   Score = supported_claims / total_claims
"""
    raw = ask(prompt)
    try:
        data = json.loads(re.sub(r"```json|```", "", raw).strip())
        return min(1.0, max(0.0, float(data.get("score", 0.5))))
    except Exception:
        return 0.5


def eval_answer_relevance(question: str, answer: str) -> float:
    """How well does the answer address the question? (0–1)"""
    prompt = f"""
Rate how directly and completely this answer addresses the question.
Return ONLY JSON: {{"score": 0.8, "reason": "brief reason"}}
Score 0 = completely unrelated, 1 = perfectly on-target.

Question: {question}
Answer: {answer}
"""
    raw = ask(prompt)
    try:
        data = json.loads(re.sub(r"```json|```", "", raw).strip())
        return min(1.0, max(0.0, float(data.get("score", 0.5))))
    except Exception:
        return 0.5


def eval_context_precision(question: str, retrieved_docs: list[dict]) -> float:
    """What fraction of retrieved docs are actually relevant? (0–1)"""
    if not retrieved_docs:
        return 0.0

    passages = "\n\n".join(
        f"[{i}] {d['text'][:200]}" for i, d in enumerate(retrieved_docs)
    )
    prompt = f"""
For each passage below, is it relevant to answering the question? Yes or No.
Return ONLY JSON: {{"relevance": [true, false, true, ...]}}

Question: {question}
Passages:
{passages}
"""
    raw = ask(prompt)
    try:
        data = json.loads(re.sub(r"```json|```", "", raw).strip())
        flags = data.get("relevance", [])
        return sum(flags) / len(flags) if flags else 0.5
    except Exception:
        return 0.5


def eval_context_recall(question: str, answer: str,
                        retrieved_docs: list[dict]) -> float:
    """Does the context contain all the info needed to answer? (0–1)"""
    context = " ".join(d["text"] for d in retrieved_docs)
    prompt = f"""
Rate how completely the context provides the information needed to answer the question.
Score 0 = context is missing key information, 1 = context has everything needed.
Return ONLY JSON: {{"score": 0.9, "missing": "what is still missing or null"}}

Question: {question}
Context: {context[:2000]}
Answer: {answer}
"""
    raw = ask(prompt)
    try:
        data = json.loads(re.sub(r"```json|```", "", raw).strip())
        return min(1.0, max(0.0, float(data.get("score", 0.5))))
    except Exception:
        return 0.5


def harmonic_mean(values: list[float]) -> float:
    """Harmonic mean — penalizes very low scores more than arithmetic mean."""
    values = [max(v, 1e-6) for v in values]  # avoid division by zero
    return len(values) / sum(1/v for v in values)


def evaluate_rag(question: str, answer: str,
                 retrieved_docs: list[dict]) -> RAGEvalResult:
    """Run all four evaluation metrics."""
    context = " ".join(d["text"] for d in retrieved_docs)

    faith  = eval_faithfulness(answer, context)
    relev  = eval_answer_relevance(question, answer)
    prec   = eval_context_precision(question, retrieved_docs)
    recall = eval_context_recall(question, answer, retrieved_docs)
    overall = harmonic_mean([faith, relev, prec, recall])

    return RAGEvalResult(
        question=question, answer=answer,
        faithfulness=faith, answer_relevance=relev,
        context_precision=prec, context_recall=recall,
        overall=overall
    )

print("✅ Evaluation functions defined!")

In [ ]:
# ── Run evaluation on a test set ──────────────────────────────────────────────

eval_questions = [
    "What is the total Q1 2025 revenue and which region performed best?",
    "What are the action items from the April meeting and their owners?",
    "What is the data retention policy for vector embeddings?",
    "How many days of vacation leave do employees get per year?",
]

# Evaluate both naive and hybrid+rerank pipelines
print("Running RAG evaluation (this calls the API multiple times)...\n")
eval_results = []

for q in eval_questions:
    # Get answer from hybrid RAG
    results = hybrid_search(q, DOCUMENTS, top_k=3)
    answer = ask(
        f"Answer using ONLY the context.\n\nContext:\n{build_context(results)}\n\nQuestion: {q}\nAnswer:"
    )

    # Evaluate
    eval_r = evaluate_rag(q, answer, results)
    eval_results.append(eval_r)
    print(eval_r)
    print()

In [ ]:
# ── Aggregate evaluation summary ──────────────────────────────────────────────

metrics = ["faithfulness", "answer_relevance", "context_precision",
           "context_recall", "overall"]

print("EVALUATION SUMMARY")
print("=" * 55)
print(f"{'Metric':<25} {'Mean':>8} {'Min':>8} {'Max':>8}")
print("─" * 55)

for m in metrics:
    vals = [getattr(r, m) for r in eval_results]
    print(f"  {m:<23} {np.mean(vals):>8.2f} {min(vals):>8.2f} {max(vals):>8.2f}")

print()
print("💡 Scores above 0.8 are generally considered production-ready.")
print("   Low context_precision → retrieve fewer but better chunks")
print("   Low faithfulness      → model is hallucinating beyond the context")
print("   Low context_recall    → missing key documents; check chunking strategy")

In [ ]:
# ── Compare pipelines head-to-head using evaluation ───────────────────────────

bench_q = "What is the total Q1 2025 revenue and which region grew the fastest?"

pipelines = {
    "Naive (vector only)": lambda q: naive_rag(q, top_k=3),
    "Hybrid (BM25 + vector)": lambda q: hybrid_rag(q, top_k=3),
}

print(f"Benchmarking question: {bench_q}\n")
print(f"{'Pipeline':<28} {'Faith':>7} {'Relev':>7} {'Prec':>7} {'Recall':>7} {'Overall':>8}")
print("─" * 68)

for name, pipeline_fn in pipelines.items():
    out = pipeline_fn(bench_q)
    ev  = evaluate_rag(bench_q, out["answer"], out["retrieved"])
    print(f"  {name:<26} "
          f"{ev.faithfulness:>7.2f} "
          f"{ev.answer_relevance:>7.2f} "
          f"{ev.context_precision:>7.2f} "
          f"{ev.context_recall:>7.2f} "
          f"{ev.overall:>8.2f}")

---

## 🏭 Section 8: Production RAG Pipeline

Now let's assemble everything into a single, configurable `RAGPipeline` class that incorporates every technique we've built:

```
User Query
    ↓
① Query Rewriting  (expand and clarify)
    ↓
② Hybrid Retrieval  (BM25 + vector + RRF)
    ↓ [top-K × 2 candidates]
③ Re-Ranking  (LLM-as-judge, reduce to top-K)
    ↓
④ Grounded Generation  (citations, gaps detection)
    ↓
⑤ Evaluation  (faithfulness, relevance, precision)
    ↓
Structured Answer + Sources + Quality Score
```

In [ ]:
@dataclass
class RAGConfig:
    """Configuration for the production RAG pipeline."""
    # Retrieval
    retrieve_k: int   = 8        # candidates to retrieve before re-ranking
    final_k: int      = 3        # chunks to send to the LLM
    rrf_k: int        = 60       # RRF smoothing constant

    # Features (toggle on/off for ablation studies)
    use_query_rewriting: bool = True
    use_multi_query: bool     = False  # slower but broader
    use_hyde: bool            = False  # good for keyword-mismatch queries
    use_reranking: bool       = True
    use_citations: bool       = True
    run_evaluation: bool      = False  # expensive — use during development


class RAGPipeline:
    """Production-grade Retrieval-Augmented Generation pipeline."""

    def __init__(self, documents: list[dict], config: RAGConfig = None):
        self.docs   = documents
        self.config = config or RAGConfig()
        self._build_bm25()

    def _build_bm25(self):
        tokenized = [tokenize(d["text"]) for d in self.docs]
        self._bm25 = BM25Okapi(tokenized)

    # ── Stage 1: Query Processing ──────────────────────────────────────────────
    def _process_query(self, question: str) -> list[str]:
        queries = [question]

        if self.config.use_query_rewriting:
            rewritten = rewrite_query(question)
            if rewritten != question:
                queries = [rewritten]

        if self.config.use_multi_query:
            queries += generate_multi_query(queries[0], n=2)

        if self.config.use_hyde:
            hypo = generate_hypothetical_document(question)
            queries.append(hypo)

        return list(dict.fromkeys(queries))  # deduplicate, preserve order

    # ── Stage 2: Retrieval ────────────────────────────────────────────────────
    def _retrieve(self, queries: list[str]) -> list[dict]:
        all_vector, all_bm25 = [], []
        for q in queries:
            all_vector.extend(vector_search(q, self.docs, self.config.retrieve_k))
            all_bm25.extend(bm25_search(q, self.docs, self.config.retrieve_k))

        # Deduplicate within each list (keep best score)
        def dedup(lst):
            seen, out = {}, []
            for item in lst:
                if item["id"] not in seen or item["score"] > seen[item["id"]]["score"]:
                    seen[item["id"]] = item
                    out.append(item)
            return out

        return reciprocal_rank_fusion(
            [dedup(all_vector), dedup(all_bm25)],
            k=self.config.rrf_k
        )[:self.config.retrieve_k]

    # ── Stage 3: Re-Ranking ───────────────────────────────────────────────────
    def _rerank(self, question: str, candidates: list[dict]) -> list[dict]:
        if self.config.use_reranking:
            return llm_rerank(question, candidates, top_k=self.config.final_k)
        return candidates[:self.config.final_k]

    # ── Stage 4: Generation ───────────────────────────────────────────────────
    def _generate(self, question: str, docs: list[dict]) -> dict:
        if self.config.use_citations:
            ga = grounded_generate(question, docs)
            return {"answer": ga.answer, "citations": ga.citations,
                    "has_gaps": ga.has_gaps, "confidence": ga.confidence}
        else:
            context = build_context(docs)
            answer = ask(
                f"Answer using ONLY the context.\n\nContext:\n{context}"
                f"\n\nQuestion: {question}\nAnswer:"
            )
            return {"answer": answer, "citations": [], "has_gaps": False, "confidence": "unknown"}

    # ── Main entry point ──────────────────────────────────────────────────────
    def query(self, question: str, verbose: bool = False) -> dict:
        """Run the full RAG pipeline for a question."""
        if verbose:
            print(f"\n{'='*65}")
            print(f"Q: {question}")
            print(f"{'='*65}")

        # Stage 1
        queries = self._process_query(question)
        if verbose:
            print(f"[1] Queries ({len(queries)}): {queries}")

        # Stage 2
        candidates = self._retrieve(queries)
        if verbose:
            print(f"[2] Retrieved {len(candidates)} candidates: "
                  f"{[c['id'] for c in candidates]}")

        # Stage 3
        final_docs = self._rerank(question, candidates)
        if verbose:
            print(f"[3] After re-ranking: {[d['id'] for d in final_docs]}")

        # Stage 4
        generation = self._generate(question, final_docs)
        if verbose:
            print(f"[4] Generated answer ({generation['confidence']} confidence)")

        result = {
            "question": question,
            "queries_used": queries,
            "retrieved": final_docs,
            **generation
        }

        # Stage 5 (optional)
        if self.config.run_evaluation:
            ev = evaluate_rag(question, generation["answer"], final_docs)
            result["evaluation"] = ev
            if verbose:
                print(f"[5] Eval: overall={ev.overall:.2f} "
                      f"faith={ev.faithfulness:.2f} relev={ev.answer_relevance:.2f}")

        return result

print("✅ RAGPipeline class defined!")

In [ ]:
# ── Instantiate and run the production pipeline ───────────────────────────────

config = RAGConfig(
    retrieve_k=6,
    final_k=3,
    use_query_rewriting=True,
    use_reranking=True,
    use_citations=True,
    run_evaluation=False,   # set True to score every response (slower)
)

rag = RAGPipeline(documents=DOCUMENTS, config=config)

# ── Run a diverse set of questions ────────────────────────────────────────────
production_questions = [
    "What is the total Q1 2025 revenue and which region had the best growth?",
    "What action items came out of the April 15 meeting and who owns them?",
    "How much does it cost to use the /v1/search API endpoint per month?",
    "How many days of sick leave and vacation leave do employees get?",
    "What are the security standards for stored embeddings?",
]

for q in production_questions:
    result = rag.query(q, verbose=True)
    print(f"\nAnswer: {result['answer']}")
    if result["citations"]:
        print(f"Sources: {[c['ref'] + ':' + c['source'] for c in result['citations'][:3]]}")
    if result["has_gaps"]:
        print(f"⚠️  Answer may be incomplete")

In [ ]:
# ── Ablation study: which features matter most? ───────────────────────────────
# Toggle features on/off to see their impact.

ablation_question = "What is the salary of engineers at the company?"

ablation_configs = {
    "Baseline (vector only)": RAGConfig(
        use_query_rewriting=False, use_reranking=False, use_citations=False
    ),
    "+ Query Rewriting": RAGConfig(
        use_query_rewriting=True, use_reranking=False, use_citations=False
    ),
    "+ Reranking": RAGConfig(
        use_query_rewriting=True, use_reranking=True, use_citations=False
    ),
    "+ Citations (Full)": RAGConfig(
        use_query_rewriting=True, use_reranking=True, use_citations=True
    ),
}

print(f"Ablation question: {ablation_question}\n")
for config_name, cfg in ablation_configs.items():
    pipeline = RAGPipeline(DOCUMENTS, cfg)
    result = pipeline.query(ablation_question)
    print(f"[{config_name}]")
    print(f"  Docs retrieved: {[r['id'] for r in result['retrieved']]}")
    print(f"  Answer: {result['answer'][:200]}")
    print()

---

## 🎉 Summary & Course Conclusion

### What You Built in This Session

| Component | Technique | Benefit |
|---|---|---|
| **Naive RAG** | Vector search + direct generation | Baseline |
| **Query Rewriting** | LLM rephrasing + multi-query expansion | Fixes vague / ambiguous questions |
| **HyDE** | Hypothetical document embeddings | Fixes keyword-mismatch between Q & docs |
| **Hybrid Search** | BM25 + vector + RRF fusion | Best of semantic AND exact-match retrieval |
| **Re-Ranking** | LLM-as-judge on top-K candidates | Precision: send only the best to LLM |
| **Grounded Generation** | Citation-enforced prompting | Verifiability + hallucination prevention |
| **Evaluation** | Faithfulness / Relevance / Precision / Recall | Measure and iterate |

### Decision Guide: Which Techniques to Use?

```
📌 Always use:    Hybrid Search (BM25 + Vector) + RRF
📌 Almost always: Query Rewriting + Grounded generation with citations
📌 When needed:   HyDE (keyword mismatch), Re-ranking (precision-critical)
📌 During dev:    Evaluation suite (faithfulness, relevance, precision)
```

### The Full 4-Session Learning Arc

| Session | Topic | What You Built |
|---|---|---|
| 1 | OpenAI API Basics | Chat, embeddings, function calling |
| 2 | Prompt Engineering | Zero-shot, CoT, self-consistency, chaining |
| 3 | Document Processing | Parse → Chunk → Embed → Store (S3 Vectors) |
| **4** | **RAG** | **Query → Retrieve → Rerank → Generate → Evaluate** |

### 🚀 Where to Go Next

- **Agentic RAG** — LLM decides when and what to retrieve (ReAct, tool use)
- **Multi-hop RAG** — Answer questions that require chaining multiple retrievals
- **Corrective RAG (CRAG)** — Self-evaluate retrieved docs, re-query if low quality
- **Graph RAG** — Retrieve from knowledge graphs, not just flat documents
- **Fine-tuned Embeddings** — Train domain-specific embedding models
- **RAGAS Framework** — Production-grade evaluation: `pip install ragas`

### 📚 Key References
- 📄 Lewis et al. (2020) — [Original RAG paper](https://arxiv.org/abs/2005.11401)
- 📄 Gao et al. (2022) — [HyDE paper](https://arxiv.org/abs/2212.10496)
- 📄 Wang et al. (2024) — [Best Practices for RAG](https://arxiv.org/abs/2407.01219)
- 📖 [RAGAS Documentation](https://docs.ragas.io)
- 📖 [AWS S3 Vectors Guide](https://docs.aws.amazon.com/AmazonS3/latest/userguide/s3-vectors.html)